In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('diabetic_data.csv')
print(f"Starting shape: {df.shape}")

Starting shape: (101766, 50)


In [2]:
df = df.drop(columns=['max_glu_serum', 'A1Cresult'])
print(f"Shape after dropping sparse columns: {df.shape}")

Shape after dropping sparse columns: (101766, 48)


In [3]:
for col in df.columns:
    count = (df[col] == '?').sum()
    if count > 0:
        print(f"{col}: {count} question marks")

race: 2273 question marks
weight: 98569 question marks
payer_code: 40256 question marks
medical_specialty: 49949 question marks
diag_1: 21 question marks
diag_2: 358 question marks
diag_3: 1423 question marks


In [4]:
print(f"Weight ? count: {(df['weight'] == '?').sum()}")
print(f"Weight ? percentage: {(df['weight'] == '?').sum()/len(df)*100:.1f}%")

Weight ? count: 98569
Weight ? percentage: 96.9%


In [5]:
df = df.drop(columns=['weight', 'payer_code'])
print(f"Shape after dropping weight and payer_code: {df.shape}")

Shape after dropping weight and payer_code: (101766, 46)


In [6]:
df['race'] = df['race'].replace('?', df['race'].mode()[0])
df['medical_specialty'] = df['medical_specialty'].replace('?', 'Unknown')

print("race ? remaining:", (df['race'] == '?').sum())
print("medical_specialty ? remaining:", (df['medical_specialty'] == '?').sum())

race ? remaining: 0
medical_specialty ? remaining: 0


In [7]:
print("diag_1 sample values:")
print(df['diag_1'].value_counts().head(10))

diag_1 sample values:
diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
427    2766
491    2275
715    2151
682    2042
434    2028
Name: count, dtype: int64


In [8]:
df['diag_1'] = df['diag_1'].replace('?', 'Unknown')
df['diag_2'] = df['diag_2'].replace('?', 'Unknown')
df['diag_3'] = df['diag_3'].replace('?', 'Unknown')

print("diag_1 ? remaining:", (df['diag_1'] == '?').sum())
print("diag_2 ? remaining:", (df['diag_2'] == '?').sum())
print("diag_3 ? remaining:", (df['diag_3'] == '?').sum())

diag_1 ? remaining: 0
diag_2 ? remaining: 0
diag_3 ? remaining: 0


In [9]:
df = df.drop(columns=['encounter_id', 'patient_nbr'])
print(f"Shape now: {df.shape}")

Shape now: (101766, 44)


In [10]:
print(f"Remaining ? values across all columns:")
for col in df.columns:
    count = (df[col] == '?').sum()
    if count > 0:
        print(f"{col}: {count}")
print("Done checking!")

Remaining ? values across all columns:
Done checking!


In [11]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Text columns remaining: {len(categorical_cols)}")
print(categorical_cols)

Text columns remaining: 33
['race', 'gender', 'age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


C:\Users\DELL-PV\AppData\Local\Temp\ipykernel_11316\3133745404.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


In [12]:
df['readmitted'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

print("Target column distribution:")
print(df['readmitted'].value_counts())
print(f"\n1 = readmitted within 30 days")
print(f"0 = not readmitted within 30 days")

Target column distribution:
readmitted
0    90409
1    11357
Name: count, dtype: int64

1 = readmitted within 30 days
0 = not readmitted within 30 days


In [13]:
# Remove readmitted from the list since it's already numeric
categorical_cols = [col for col in categorical_cols if col != 'readmitted']

# Encode all text columns to numbers
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Shape after encoding: {df.shape}")
print(f"All columns are now numbers: {df.select_dtypes(include=['object']).shape[1] == 0}")

Shape after encoding: (101766, 2402)
All columns are now numbers: True


In [14]:
print("=== CLEAN DATASET SUMMARY ===")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nTarget distribution:")
print(df['readmitted'].value_counts())

=== CLEAN DATASET SUMMARY ===
Total rows: 101766
Total columns: 2402
Missing values: 0

Target distribution:
readmitted
0    90409
1    11357
Name: count, dtype: int64


In [15]:
df.to_csv('diabetic_data_clean.csv', index=False)
print("Clean dataset saved!")

Clean dataset saved!


In [16]:
df_check = pd.read_csv('diabetic_data_clean.csv')
print(f"Reloaded shape: {df_check.shape}")
print(f"Missing values: {df_check.isnull().sum().sum()}")
print("Week 2 complete!")

Reloaded shape: (101766, 2402)
Missing values: 0
Week 2 complete!
